# 05 — Geography, boundaries and the IEC Atlas

**The Democratic Funnel** · Team UL · DIRISA Student Datathon Challenge 2026

**Owner:** Role E  **Date:** 25 September 2026

---

## Purpose

Three jobs, all feeding the master table:

1. **Municipal geometry** — the map layer the dashboard draws, plus area and a neighbour
   list for the spatial-lag feature.
2. **The 2011 → current crosswalk** — municipal boundaries were redrawn in 2016, so 2011
   election results cannot be joined to current municipality codes directly. We derive the
   mapping from the geometry itself rather than compiling it by hand.
3. **IEC Atlas data** — registration *activity* (new registrations, re-registrations,
   voting-district moves) per municipality, which is published nowhere else.

## Outputs

| File | Contents |
|---|---|
| `data/interim/boundaries.gpkg` | 213 municipalities, 2026 MDB boundaries, with area |
| `data/interim/neighbours.csv` | 1 100 directed border pairs |
| `data/interim/muni_crosswalk_2011.csv` | 234 old codes → 213 current, with area shares |
| `data/interim/atlas_registration_activity_2016.csv` | Registration activity per municipality, 2016 |
| `data/interim/atlas_registration_activity_2011.csv` | Same for 2011, on 2011 boundaries |
| `data/interim/atlas_layers.csv` | The full IEC Atlas layer catalogue (181 layers) |

## Sources

| Source | Where |
|---|---|
| MDB Local Municipalities 2026 (and 2011, 2016, 2018 layers) | MDB Spatial Knowledge Hub, `spatialhub-mdb-sa.opendata.arcgis.com` |
| IEC LGE Atlas | `atlas.elections.org.za/lgeatlas/` — data served over WFS (see §3) |

Both logged in `docs/sources.csv` with URLs and access dates.

## 0. Setup

Runs the shared project configuration: resolves paths, creates folders, fixes the random seed.

In [ ]:
import sys
from pathlib import Path

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "src" / "config.py").exists())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src import config
from src.config import RAW, INTERIM, PROCESSED, FIGURES, SEED

import json, time, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
print(config.describe())

---
## 1. Municipal boundaries

Source: **MDB Local Municipalities 2026** from the Municipal Demarcation Board's Spatial
Knowledge Hub, downloaded as GeoJSON.

We take the *local municipalities* layer, which on inspection already includes the eight
metropolitan municipalities — `CATEGORY` is `A` for metros and `B` for local municipalities.
District municipalities are **not** in this layer, which is what we want: every voter already
sits inside a local municipality or metro, so including districts as rows would count the
same people twice.

In [ ]:
gdf = gpd.read_file(RAW / "boundaries" / "MDB_L_ocal_Municipalities_2026.geojson")

print(gdf.shape, gdf.crs)
print(gdf.columns.tolist())
gdf.head(3)

**Result:** 213 rows × 17 columns, CRS **EPSG:4326** (WGS84 lat/lon).

The join key is **`CAT_B`** — values such as `BUF`, `NMA`, `EC101` — which matches the
municipality codes used in the IEC result files and the registration dashboard. `MUNICNAME`
carries the name, `CATEGORY` the A/B metro-vs-local split, and `Area_sqkm` MDB's own area
calculation.

We rename to the project's standard names (`muni_code`, `muni_name`, …) and drop the MDB
administrative fields we do not use.

In [ ]:
gdf = gdf.rename(columns={"CAT_B": "muni_code", "MUNICNAME": "muni_name",
                          "PROVINCE": "province", "CATEGORY": "muni_category",
                          "DISTRICT": "district_code", "Area_sqkm": "area_km2"})
gdf = gdf[["muni_code", "muni_name", "province", "muni_category",
           "district_code", "area_km2", "geometry"]]

print(gdf.shape)
gdf.head(3)

### Checks

Four things must hold before this layer is used anywhere else.

In [ ]:
print("duplicate codes :", gdf["muni_code"].duplicated().sum())
print("category counts :", gdf["muni_category"].value_counts().to_dict())
print(f"total area      : {gdf['area_km2'].sum():,.0f} km2   (South Africa ~1 220 000 km2)")

# cross-check MDB's area against our own calculation
gdf["area_km2_calc"] = gdf.to_crs(epsg=3857).area / 1e6
print("\nMDB vs computed area (Web Mercator distorts area at SA latitudes,")
print("so we keep MDB's figure as authoritative):")
print(gdf[["area_km2", "area_km2_calc"]].describe().round(0))

ax = gdf.plot(figsize=(7, 7), edgecolor="white", linewidth=0.2, color="#4a5059")
ax.set_axis_off(); ax.set_title("213 municipalities, MDB 2026 boundaries")
plt.show()

**All four pass.** No duplicate codes, 8 metros and 205 local municipalities, the area
total matches South Africa, and the plot is the country with no stray polygons.

We keep `area_km2_calc` alongside MDB's `area_km2` only as an audit trail. Web Mercator
(EPSG:3857) badly distorts area at South African latitudes, so **MDB's own figure is the one
we use** in any calculation.

---
## 2. Neighbour list

Which municipalities share a border. This feeds the **spatial lag** feature: for each
municipality, the average turnout of the municipalities around it.

Low participation clusters geographically — a municipality surrounded by low-turnout
neighbours is itself more likely to have low turnout, for reasons unconnected to its own
registration figures. The spatial lag lets the model capture that regional pattern instead of
treating each municipality as independent.

`predicate="touches"` returns every pair of polygons sharing a boundary. Each border appears
**twice**, once in each direction (A→B and B→A), which is what we want: every municipality
needs its own complete list of neighbours.

In [ ]:
nb = gpd.sjoin(gdf[["muni_code", "geometry"]],
               gdf[["muni_code", "geometry"]],
               predicate="touches")[["muni_code_left", "muni_code_right"]]
nb = nb[nb.muni_code_left != nb.muni_code_right].drop_duplicates()

print(len(nb), "directed border pairs")
print(nb.groupby("muni_code_left").size().describe().round(2).to_string())

nb.to_csv(INTERIM / "neighbours.csv", index=False)

**Result:** 1 100 directed pairs (≈550 actual borders), a mean of **5.16** neighbours per
municipality, minimum **1**, maximum **10**.

No municipality is isolated, which confirms the geometry is sound — a zero would mean either an
island or a broken polygon.

---
## 3. The IEC Atlas

The IEC publishes an *Atlas of Results* at `atlas.elections.org.za/lgeatlas/`. It presents
itself as a map viewer with no download option, and the competition brief lists it as one of
the scattered sources that are "rarely combined into a single coherent picture".

Inspecting the page's network requests showed it is served by **GeoServer**, and that the
server's **WFS endpoint is open**. That means every thematic layer behind the map can be
requested as GeoJSON and analysed directly, rather than read off a picture.

Two entry points:

| What | Where |
|---|---|
| Layer catalogue | `lgeatlas/Scripts/report_layers.js` — JSON despite the `.js` extension |
| Feature data | `https://atlas.elections.org.za:8443/geoserver/wfs` (WFS 1.0.0) |

This is the single most valuable thing in this notebook: it gives us municipal **registration
activity**, which is published neither on the registration dashboard nor in the downloadable
result files.

### 3.1 The layer catalogue

In [ ]:
js = requests.get("https://atlas.elections.org.za/lgeatlas//Scripts/report_layers.js",
                  timeout=60).text
cat = json.loads(js[js.index("{"):])

rows = [{"theme": h["label"], "group": g["label"], "year": l["label"],
         "layer": l["lyrurl"], "key": l.get("titlekey"),
         "cols": [c["name"] for c in l.get("columns", [])]}
        for h in cat["grouplayers"] for g in h["groups"] for l in g["layers"]]

layers = pd.DataFrame(rows)
layers.to_csv(INTERIM / "atlas_layers.csv", index=False)

print(len(layers), "layers")
layers["theme"].value_counts()

**181 layers**, covering voting-district delimitation, voter registration, voting-station
distribution and results, for elections from 2000 to 2021.

Three groups are directly useful to the funnel:

- **Total registration activity by municipality** (2011, 2016) — new registrations,
  re-registrations in the same voting district, and re-registrations in a different one,
  split by registration weekend
- **Geography of new voting districts by municipality** — how many voting districts were added
  each cycle
- **Voter registration density by voting district** (2011, 2016, 2021)

### 3.2 Pulling a layer over WFS

In [ ]:
GS = "https://atlas.elections.org.za:8443/geoserver/wfs"

def get_layer(type_name, max_features=None, timeout=300):
    """Request one Atlas layer as GeoJSON. Pass max_features to inspect a schema cheaply."""
    params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
              "typeName": type_name, "outputFormat": "application/json"}
    if max_features:
        params["maxFeatures"] = max_features
    r = requests.get(GS, params=params, timeout=timeout)
    r.raise_for_status()
    return r

(RAW / "atlas").mkdir(exist_ok=True)

### 3.3 Registration activity, 2016

Layer `LGEAtlas:Total registration combined 2016`. Roughly 28 MB with geometry attached; we
keep the attributes and discard the geometry, since we already have boundaries from MDB.

Column naming follows a pattern: plain columns are **registration weekend 1**, the `X` prefix is
**weekend 2**, and the `3` suffix is the **combined** total.

| Column | Meaning |
|---|---|
| `REGPOP` / `XPOP` / `POP3` | Registered population at each point |
| `NEW_REG` / `XNEWREG` / `NEWREG3` | **New** registrations — first time on the roll |
| `VD_MOVE` / `XVDMOVE` / `VDM3` | Re-registered in a *different* voting district (moved) |
| `REREG` / `REREG2` / `REREG3` | Re-registered in the *same* voting district |
| `TOTALW1` / `XTOTAL` / `TOTAL3` | Total activity |

In [ ]:
r = get_layer("LGEAtlas:Total registration combined 2016")
(RAW / "atlas" / "registration_activity_2016.json").write_text(r.text, encoding="utf-8")
print(r.headers.get("content-type"), f"{len(r.content)/1e6:.1f} MB")

ra = gpd.read_file(RAW / "atlas" / "registration_activity_2016.json")
print(ra.shape)
print(ra.columns.tolist())

In [ ]:
ra = ra.rename(columns={"MUNICCODE": "muni_code"})
keep = ["muni_code", "REGPOP", "NEW_REG", "VD_MOVE", "REREG", "TOTALW1",
        "XPOP", "XNEWREG", "XVDMOVE", "REREG2", "XTOTAL",
        "POP3", "NEWREG3", "VDM3", "REREG3", "TOTAL3"]
ra2016 = pd.DataFrame(ra[keep]).copy()

# codes match the MDB 2026 boundaries exactly - no crosswalk needed 2016 -> 2026
assert set(ra2016["muni_code"]) == set(gdf["muni_code"])

ra2016.to_csv(INTERIM / "atlas_registration_activity_2016.csv", index=False)
print(ra2016.shape)
ra2016.head(3)

**Finding — no crosswalk needed after 2016.** The assertion passes: all 213 municipality
codes in the 2016 Atlas layer are identical to those in the 2026 MDB boundaries. Municipal
boundaries have therefore been stable across 2016 → 2021 → 2026, and only the 2011 election
needs a crosswalk.

### 3.4 Registration activity, 2011

Layer `LGEAtlas:Total_registration_Activity_(Combined) 2011`. This one is on **2011
boundaries**, so it has 234 municipalities and its codes are *not* current.

In [ ]:
time.sleep(3)   # be gentle with the IEC's server

r11 = get_layer("LGEAtlas:Total_registration_Activity_(Combined) 2011")
(RAW / "atlas" / "registration_activity_2011.json").write_text(r11.text, encoding="utf-8")

ra11 = gpd.read_file(RAW / "atlas" / "registration_activity_2011.json")
print(ra11.shape)
print(ra11.columns.tolist())

# CODE holds the municipality code; SMUNICIPAL holds "CODE - NAME"
for c in ["CODE", "SMUNICIPAL"]:
    print(f"{c:12s}", ra11[c].head(3).tolist())

In [ ]:
ra2011 = pd.DataFrame(ra11[["CODE", "SMUNICIPAL",
                            "TOTAL_ACTI", "TOTAL_ACT1", "TOTALACTIV"]]).copy()
ra2011 = ra2011.rename(columns={"CODE": "muni_code_2011",
                                "SMUNICIPAL": "muni_name_2011"})
ra2011.to_csv(INTERIM / "atlas_registration_activity_2011.csv", index=False)
print(ra2011.shape)
ra2011.head(3)

The key column is deliberately named **`muni_code_2011`**, not `muni_code`. These are 2011
codes and merging them directly onto current data would be wrong — see §4. They become usable
only after the crosswalk is applied.

---
## 4. The 2011 → current crosswalk

### Why it is needed

Municipal boundaries were redrawn for the 2016 election and several municipalities were merged.
South Africa had **234** municipalities in 2011 and has **213** today.

Worse than a simple count change: **codes were reused for different areas.** The clearest case
is `EC101`.

| Election | `EC101` is | Registered voters |
|---|---|---|
| 2011 | **Camdeboo** | 23 164 |
| 2016 → | **Dr Beyers Naude** = Camdeboo + Ikwezi + Baviaans | 40 887 |

A naive merge on `muni_code` would pair Camdeboo's 2011 figures with Dr Beyers Naude's 2016
figures and read the result as **72% registration growth** in one municipality — when in fact
two other municipalities were added to it.

### The method

Rather than compiling the mapping by hand from demarcation notices, we derive it from the
geometry: intersect the 2011 boundaries with the current ones and measure how much of each old
municipality's area now falls inside each current one. That area share becomes the weight used
to apportion counts.

This is faster, reproducible, and carries its own evidence.

In [ ]:
# 2011 boundaries come from the Atlas layer we already pulled
g11 = ra11.to_crs(epsg=3857)[["CODE", "SMUNICIPAL", "geometry"]]
cur = gdf.to_crs(epsg=3857)[["muni_code", "muni_name", "geometry"]]

ov = gpd.overlay(g11, cur, how="intersection")
ov["overlap_km2"] = ov.area / 1e6
ov["share"] = ov["overlap_km2"] / ov.groupby("CODE")["overlap_km2"].transform("sum")

print(len(ov), "raw intersection pieces")

### Slivers vs genuine splits

Redrawing a boundary by a few metres creates tiny intersection pieces that are artefacts, not
real overlaps. We drop anything below **1%** of an old municipality's area.

What remains above that threshold is either a clean one-to-one mapping or a genuine split.

In [ ]:
multi = ov[ov["share"] >= 0.01].groupby("CODE").size()
split_codes = multi[multi > 1].index.tolist()

sub = ov[ov["CODE"].isin(split_codes) & (ov["share"] >= 0.01)]
print(sub.sort_values(["CODE", "share"], ascending=[True, False])
        [["CODE", "SMUNICIPAL", "muni_code", "muni_name", "share"]]
        .to_string(index=False))

**Nine municipalities have more than one successor above the 1% threshold, and five are
genuine splits** where assigning the whole municipality to its largest successor would discard a
substantial share:

| 2011 municipality | Split between | Shares |
|---|---|---|
| **FS203 Ngwathe** | Ngwathe / Metsimaholo | 0.51 / 0.49 |
| **LIM342 Mutale** | Musina / Collins Chabane / Thulamela | 0.39 / 0.32 / 0.29 |
| **KZN283 Ntambanana** | Mthonjaneni / uMhlathuze / Mfolozi | 0.54 / 0.41 / 0.05 |
| **LIM343 Thulamela** | Collins Chabane / Thulamela | 0.74 / 0.26 |
| **LIM352 Aganang** | Polokwane / Blouberg / Molemole | 0.70 / 0.15 / 0.15 |

The other four (KZN262, KZN293, LIM333, KZN211, KZN273) are lopsided at 77–89% and a
largest-successor mapping would be adequate — but since we apportion by share anyway, they are
handled correctly without a special case.

In [ ]:
cw = ov[ov["share"] >= 0.01].copy()
cw = cw.rename(columns={"CODE": "muni_code_2011", "SMUNICIPAL": "name_2011",
                        "muni_code": "muni_code_current"})
cw = cw[["muni_code_2011", "name_2011", "muni_code_current", "muni_name", "share"]]
cw.to_csv(INTERIM / "muni_crosswalk_2011.csv", index=False)
print(cw.shape)
cw.head(5)

### Verification

In [ ]:
print(cw["muni_code_2011"].nunique(), "of 234 old codes covered")

s = cw.groupby("muni_code_2011")["share"].sum()
print("\nshares per old municipality (should sum to ~1):")
print(s.describe().round(4).to_string())
print("\nany losing more than 2% to the sliver threshold:", (s < 0.98).sum())

**All 234 old codes are covered.** Shares sum to a mean of **0.9990**, minimum **0.9856** —
so the worst case still retains 98.6% of its area after dropping sub-1% slivers.

### How to use it

```python
# counts are apportioned by share, then summed onto current codes
m = df_2011.merge(cw, left_on="muni_code", right_on="muni_code_2011", how="left")
for c in ["registered", "votes_cast", "valid_votes", "spoilt"]:
    m[c] = m[c] * m["share"]
out = m.groupby("muni_code_current", as_index=False)[counts].sum()

# rates are recomputed AFTER summing - never apportioned
out["turnout"] = out["votes_cast"] / out["registered"]
```

**The rule about rates matters.** Turnout is a ratio: you cannot take 51% of one percentage and
add it to 49% of another. Apportion the counts, sum them, then divide. Doing it the other way
produces numbers that look plausible and are wrong.

Applied to the 2011 results in `01_results_lge`, this crosswalk preserves the national totals
(23 654 347 → 23 623 818 registered voters, a 0.13% loss to slivers) and leaves national turnout
unchanged at **0.5745**. `EC101` in 2011 becomes **38 211** registered voters — Camdeboo plus
Ikwezi plus Baviaans — instead of Camdeboo's 23 164, which sits sensibly against the 40 887
recorded for the merged municipality in 2016.

---
## 5. Save

In [ ]:
gdf.to_file(INTERIM / "boundaries.gpkg", driver="GPKG")

for f in sorted(INTERIM.glob("*")):
    if f.is_file() and f.name != ".gitkeep":
        print(f"{f.name:45s} {f.stat().st_size/1024:8.1f} KB")

---
## 6. Summary and limitations

### What this notebook established

1. **The IEC Atlas serves its data over an open WFS endpoint**
   (`atlas.elections.org.za:8443/geoserver/wfs`), so its 181 thematic layers can be pulled as
   GeoJSON rather than read off a map image. The layer catalogue is at
   `lgeatlas/Scripts/report_layers.js`.
2. **Registration *activity* — new registrations, re-registrations, voting-district moves — is
   published only through the Atlas**, not on the registration dashboard and not in the
   downloadable result files. It measures how much registration *effort* occurred in each
   municipality, which is a direct input to the registration leak.
3. **Municipality codes are identical between the 2016 Atlas layer and the 2026 MDB boundaries**
   (213 of 213), so no crosswalk is required for 2016 → 2021 → 2026.
4. **The 2011 → current crosswalk was derived from geometry**, not compiled by hand: all 234 old
   codes are covered, shares sum to a mean of 0.999, and five genuinely split municipalities are
   apportioned rather than assigned wholesale to one successor.

### Limitations

| Limitation | Statement |
|---|---|
| **Area share is a proxy for population share** | Where a split municipality's population was unevenly distributed across the parts, apportioned counts are approximate. It is a better approximation than assigning the whole municipality to one successor; the alternative would require voting-district-level population data for 2011, which is not published. |
| **Sliver threshold** | Overlaps below 1% of an old municipality's area are treated as boundary-redrawing artefacts and dropped. This loses 0.13% of 2011 registered voters nationally; the worst-affected municipality retains 98.6% of its area. |
| **Atlas vintage** | The municipal Atlas is the 2016 edition. Its registration-activity layers cover 2011 and 2016 only — there is no equivalent for 2021 or 2026. |
| **Atlas activity is not a forecast feature** | Because it exists only for 2011 and 2016, it cannot feed a 2026 prediction: the feature would be absent for the row being predicted. It is used for exploratory analysis and the funnel narrative, not in the model's feature set. |
| **Area from MDB** | We use MDB's published `Area_sqkm` rather than our own calculation. Reprojecting to Web Mercator distorts area at South African latitudes; our computed column is retained only as an audit trail. |

### Where this feeds

- `boundaries.gpkg` → the dashboard map, and `area_km2` / `pop_density` in the master table
- `neighbours.csv` → `spatial_lag_turnout` in the master table
- `muni_crosswalk_2011.csv` → applied to the 2011 election results in `01_results_lge`
- `atlas_registration_activity_*.csv` → exploratory analysis in `07_eda`